# Session 8 Lab: A Research Team of Agents (LangGraph)

**Project 5 · Research team (multi-agent)** · GenAI & Agentic AI Workshop

You'll build a small team:

- 🔎 **Researcher**: searches Wikipedia (a real tool) and writes notes with sources
- ✍️ **Writer**: turns the notes into a short article for first-year students
- 🧐 **Reviewer**: checks the draft and sends it back if something is wrong
- 🙋 **You**: approve the final article, or ask for changes (human-in-the-loop)

First as plain Python, then as a **LangGraph** graph.

**Before you start:** you need the `GEMINI_API_KEY` secret (🔑 Secrets, Notebook access ON). Run each cell with **Shift + Enter**.

In [ ]:
!pip install -q langgraph langchain-google-genai

In [ ]:
import json, requests
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

MODEL = "gemini-2.5-flash"   # any current free model from Google AI Studio works
llm = ChatGoogleGenerativeAI(model=MODEL, api_key=userdata.get("GEMINI_API_KEY"), temperature=0.3)

print(llm.invoke("Say hello to the workshop in 5 words.").text)

## Step 1: The researcher's tool

A real, free tool: search Wikipedia and return short summaries with links. The `@tool` decorator turns a Python function into a tool LangChain can describe to the model (like the declarations in Session 7, written for you).

Wikipedia limits how many requests one network can make. A whole class on the same Wi-Fi can hit that limit, so the tool waits and retries once, falls back to a few **backup facts** for our example topic, and otherwise returns an error instead of crashing (Session 7: return errors, don't crash).

In [ ]:
import time

WIKI_HEADERS = {"User-Agent": "genai-workshop-research-agent/1.0 (educational demo)"}

# Backup facts, used only if Wikipedia is too busy, so the lab still works for the example topic.
BACKUP = {("chandrayaan", "vikram", "pragyan"): [{
    "title": "Chandrayaan-3",
    "url": "https://en.wikipedia.org/wiki/Chandrayaan-3",
    "summary": ("Chandrayaan-3 is the third lunar exploration mission of the Indian Space Research Organisation (ISRO). "
                "It was launched on 14 July 2023 on an LVM3 rocket from the Satish Dhawan Space Centre, Sriharikota. "
                "Its Vikram lander touched down near the Moon's south pole on 23 August 2023, making India the fourth "
                "country to soft-land on the Moon and the first to land near the lunar south pole. The lander carried "
                "the Pragyan rover, which drove on the surface and studied the lunar soil. India now celebrates "
                "23 August as National Space Day."),
}]}

@tool
def search_wikipedia(query: str) -> list:
    """Search English Wikipedia. Returns up to 3 results, each with a title, a short summary and a URL."""
    for attempt in range(2):
        try:
            r = requests.get("https://en.wikipedia.org/w/api.php", headers=WIKI_HEADERS, timeout=10, params={
                "action": "query", "generator": "search", "gsrsearch": query, "gsrlimit": 3,
                "prop": "extracts|info", "exintro": 1, "explaintext": 1, "inprop": "url",
                "format": "json", "formatversion": 2,
            })
            if r.status_code == 200:
                pages = sorted(r.json().get("query", {}).get("pages", []), key=lambda p: p.get("index", 0))
                return [{"title": p["title"], "summary": p.get("extract", "")[:1200], "url": p.get("fullurl", "")}
                        for p in pages]
        except requests.RequestException:
            pass
        time.sleep(5)                                 # busy? wait a little and try once more
    for keywords, results in BACKUP.items():          # still busy: use backup facts if we have them
        if any(k in query.lower() for k in keywords):
            return results
    return [{"error": "Wikipedia is busy right now (too many requests). Try again in a minute."}]

for r in search_wikipedia.invoke({"query": "Chandrayaan-3"}):
    print("📄", r.get("title"), "-", r.get("url") or r.get("error"))

## Step 2: Three agents = three functions

Each agent is just an LLM call with its **own role (system prompt)** and, for the researcher, its **own tool**. Each one reads from a shared `state` dictionary and returns what it adds to it.

This is the key idea of multi-agent systems: **split one big job into specialists.**

In [ ]:
RESEARCHER_PROMPT = """You are a careful researcher for a college tech magazine.
Use the search_wikipedia tool 2 or 3 times to find facts about the topic.
Then write research notes: 6 to 10 short bullet points of key facts.
End every bullet with its source URL in brackets. Only use facts from the search results."""

def researcher(state):
    research_llm = llm.bind_tools([search_wikipedia])
    messages = [SystemMessage(RESEARCHER_PROMPT), HumanMessage(f"Topic: {state['topic']}")]
    for _ in range(5):                                   # a small, capped tool loop (Session 7!)
        ai = research_llm.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            break
        for call in ai.tool_calls:
            print(f"   🔎 researcher searches: {call['args'].get('query')}")
            result = search_wikipedia.invoke(call["args"])
            messages.append(ToolMessage(content=json.dumps(result), tool_call_id=call["id"]))
    return {"notes": ai.text}


WRITER_PROMPT = """You are a writer for a college tech magazine.
Using ONLY the research notes, write an article of about 200 words for first-year students:
a catchy title, simple English, short paragraphs, and a 'Sources' list of URLs at the end.
If there is feedback from the reviewer or the editor, fix every point."""

def writer(state):
    request = f"Topic: {state['topic']}\n\nResearch notes:\n{state['notes']}"
    if state.get("feedback"):
        request += f"\n\nPrevious draft:\n{state['draft']}\n\nFeedback to fix:\n{state['feedback']}"
    draft = llm.invoke([SystemMessage(WRITER_PROMPT), HumanMessage(request)]).text
    return {"draft": draft, "revisions": state.get("revisions", 0) + 1}

In [ ]:
from pydantic import BaseModel, Field

class Review(BaseModel):
    approved: bool = Field(description="True only if the draft needs no changes")
    feedback: str = Field(description="Short bullet list of what to fix, or 'None'")

REVIEWER_PROMPT = """You are a strict magazine editor. Check the draft against the research notes:
- Is any fact NOT in the notes? (That's a hallucination: reject.)
- Is it longer than 250 words, or hard for a first-year student to read?
- Is the Sources list missing?
Approve only if everything is fine. Otherwise give short, specific feedback."""

def reviewer(state):
    review = llm.with_structured_output(Review).invoke([
        SystemMessage(REVIEWER_PROMPT),
        HumanMessage(f"Research notes:\n{state['notes']}\n\nDraft:\n{state['draft']}"),
    ])
    print(f"   🧐 reviewer: {'approved ✅' if review.approved else 'changes needed ✏️'}")
    return {"approved": review.approved, "feedback": "" if review.approved else review.feedback}

## Step 3: Run the team as plain Python

No framework yet. Just call the agents one after another, passing the state along. This is a **sequential pipeline**.

In [ ]:
TOPIC = "Chandrayaan-3, India's mission to the Moon"   # try your own topic!

state = {"topic": TOPIC}
print("🔎 Researcher working...")
state |= researcher(state)
print("✍️  Writer working...")
state |= writer(state)
print("🧐 Reviewer working...")
state |= reviewer(state)

print("\n----- NOTES -----\n", state["notes"])
print("\n----- DRAFT -----\n", state["draft"])
print("\n----- REVIEW -----\n", "Approved!" if state["approved"] else state["feedback"])

That works, but what if the reviewer rejects the draft? We'd need a loop back to the writer, a limit on retries, a pause for a human to approve, and a way to resume later... You could write all that yourself, or use a **framework**.

## Step 4: The same team as a LangGraph graph

In LangGraph you describe your app as a **graph**:
- **State**: the shared data (a typed dictionary)
- **Nodes**: the agents (our three functions, unchanged!) plus a human step
- **Edges**: who goes next, including **conditional edges** (if/else) and loops

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

class TeamState(TypedDict, total=False):
    topic: str
    notes: str
    draft: str
    feedback: str
    approved: bool
    revisions: int

MAX_REVISIONS = 3

def human_approval(state):
    # PAUSE the graph and wait for a person (human-in-the-loop)
    answer = interrupt({"draft": state["draft"], "question": "Type 'yes' to publish, or type what to change:"})
    if answer.strip().lower() in ("yes", "y", "ok", "approve"):
        return {"approved": True, "feedback": ""}
    return {"approved": False, "feedback": f"Editor-in-chief says: {answer}"}

def after_review(state):
    if state["approved"] or state["revisions"] >= MAX_REVISIONS:
        return "human_approval"
    return "writer"                                   # loop back with the reviewer's feedback

def after_human(state):
    return END if state["approved"] else "writer"

graph = StateGraph(TeamState)
graph.add_node("researcher", researcher)
graph.add_node("writer", writer)
graph.add_node("reviewer", reviewer)
graph.add_node("human_approval", human_approval)

graph.add_edge(START, "researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer", "reviewer")
graph.add_conditional_edges("reviewer", after_review, ["writer", "human_approval"])
graph.add_conditional_edges("human_approval", after_human, ["writer", END])

app = graph.compile(checkpointer=InMemorySaver())   # the checkpointer lets the graph pause and resume

Draw it. LangGraph can draw your graph for you:

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())   # text version if the image service is unavailable

## Step 5: Run it, with you as the editor-in-chief

The graph runs until it reaches `human_approval`, then **pauses** and shows you the draft. Type `yes` to publish, or type a change (for example *"make it more fun and add one fun fact"*) and it goes back to the writer.

In [ ]:
config = {"configurable": {"thread_id": "article-1"}}   # one thread = one conversation with the team
inp = {"topic": TOPIC, "revisions": 0}

while True:
    for update in app.stream(inp, config, stream_mode="updates"):
        for node, change in update.items():
            if node == "__interrupt__":
                print("\n📝 DRAFT FOR YOUR APPROVAL:\n")
                print(change[0].value["draft"])
            else:
                print(f"✅ {node} done")
    if not app.get_state(config).next:                  # nothing left to run: finished
        break
    inp = Command(resume=input("\nYour decision (yes / what to change): "))

final = app.get_state(config).values
print(f"\n🎉 Published after {final['revisions']} draft(s).")

## Step 6: Look inside

Everything the team did is saved in the **state**, so you can debug it. This is why frameworks help: state, pausing, resuming and history come for free.

In [ ]:
final = app.get_state(config).values
print("Topic:     ", final["topic"])
print("Revisions: ", final["revisions"])
print("Approved:  ", final["approved"])
print("\nFinal article:\n")
print(final["draft"])

print("\nHistory (newest first):")
for snap in app.get_state_history(config):
    print(" ", snap.next or "(end)", "| revisions so far:", snap.values.get("revisions"))

## Stretch goals

1. **New topic:** set `TOPIC` to something from your branch ("How UPI works", "What is Kubernetes", "History of ISRO") and run Step 5 again with a new `thread_id`.
2. **Add an agent:** add a `fact_checker` node between `writer` and `reviewer` that checks each sentence against the notes.
3. **Parallel researchers (orchestrator-worker):** two researchers searching different angles at the same time. Hint: add edges `START → researcher_a` and `START → researcher_b`, both into `writer`, and make `notes` a list using `Annotated[list, operator.add]` so both results are kept.
4. **Swap the tool:** give the researcher your `search_faq` (Session 4) or `get_exam_date` (Session 7) tool instead of Wikipedia.
5. **Count the cost:** how many LLM calls did one article take? Multi-agent systems are powerful but use many more tokens.

## Done? 🎉

Save a copy and push it to your GitHub as Project 5. Paste your best article in the README!